# 05 — Live USGS earthquakes → H3 → anomaly detection

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/05_h3_earthquake_geoai.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/05_h3_earthquake_geoai.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `53f90147a7726aa5d31da69f7a0bff56fdacac0812c72f5e3bf08cf2c0c84891`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 05 — Live USGS earthquakes → H3 → anomaly detection

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=05_h3_earthquake_geoai.ipynb)**

This notebook is intentionally dynamic: it reads the USGS **all earthquakes, past 30 days** GeoJSON feed, aggregates events into H3 cells, then uses an Isolation Forest to highlight cells with unusual combinations of event count and magnitude.

Because the source feed changes continuously, record the run timestamp when using the result outside this tutorial.

> **JupyterLite compatibility:** this notebook uses `geolibre_lite.LiteMap`, which builds native GeoLibre project/layer definitions and displays them through GeoLibre's hosted `?embed=1` viewer. It deliberately avoids `geolibre.Map`, whose localhost HTTP server cannot bind a socket inside Pyodide/WebAssembly.


## Start here: the question behind the code

**Which spatial event summaries differ from the rest of this feed?**

**What you will learn:** collect → aggregate → score → challenge. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Browser machine learning: Isolation Forest scores unusual H3 feature profiles.

**Why this is useful:** A model can help focus exploration of a large event table, while the map and charts expose why a profile might be unusual.

**Where the work happens:** USGS events are downloaded; aggregation, H3 indexing and model fitting run locally in Pyodide.

![Unusual is a question to investigate. Read event coordinates, magnitudes and depths from a changing monthly feed. Represent each occupied cell with event count, magnitude and depth summaries. Assign higher values to unusual feature combinations in this sample. Inspect time window, missingness and depth before interpreting a high score.](assets/learning/05/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: Unusual is a question to investigate — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/05/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/05/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/05/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=05_h3_earthquake_geoai.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/05_h3_earthquake_geoai.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/05/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Collect (0–8 s):** Read event coordinates, magnitudes and depths from a changing monthly feed. Code landmark: `response.json()`.

2. **Aggregate (8–16 s):** Represent each occupied cell with event count, magnitude and depth summaries. Code landmark: `eq.groupby('h3')`.

3. **Score (16–24 s):** Assign higher values to unusual feature combinations in this sample. Code landmark: `-iso.score_samples(X)`.

4. **Challenge (24–32 s):** Inspect time window, missingness and depth before interpreting a high score. Code landmark: `agg.sort_values(...)`.


## See the connections

![Earthquakes: events, bins and unusual patterns: Events: USGS time, magnitude and depth records; Aggregation: H3 cells count events in a time window; Learning: Anomaly score measures unusual features; Check: Catalog outliers are not earthquake forecasts](assets/05_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])
from geolibre_lite import LiteMap as Map


### Code guide 2 — Prepare aggregation and anomaly tools

**Read the Python:** pandas stores the event table, H3 assigns spatial cells, and Isolation Forest learns unusual feature combinations. The HTTP patch adapts network calls to Pyodide without bypassing provider access rules.

**Look for:** Imported functions, with no feed downloaded or model trained yet.

**Pause and try:** Find the package responsible for geometry indexing and the one responsible for learning.


In [ ]:
import sys, requests, pandas as pd, numpy as np
if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

import h3
from sklearn.ensemble import IsolationForest


### Code guide 3 — Read events and give each a spatial index

**Read the Python:** The loop separates coordinates from properties in each GeoJSON feature. Missing magnitudes become `np.nan`, not zero. `zip(eq.lat, eq.lon)` pairs each latitude with its longitude before assigning H3 resolution 3.

**Look for:** An event table and its record count. The monthly feed changes over time.

**Pause and try:** Inspect an event's depth unit and record the feed date before comparing it with a future run.


In [ ]:
USGS = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"
response = requests.get(USGS, timeout=60)
response.raise_for_status()
feed = response.json()

rows = []
for f in feed["features"]:
    lon, lat, depth = f["geometry"]["coordinates"][:3]
    p = f["properties"]
    rows.append({
        "lon": lon, "lat": lat, "depth_km": depth,
        "mag": p.get("mag") if p.get("mag") is not None else np.nan,
        "place": p.get("place"), "time": p.get("time"),
    })

if not rows:
    raise ValueError('No earthquake records returned; adjust the feed/query before continuing.')
eq = pd.DataFrame(rows)
eq["h3"] = [h3.latlng_to_cell(lat, lon, 3) for lat, lon in zip(eq.lat, eq.lon)]
eq.head(), len(eq)


### Code guide 4 — Learn which cell profiles are unusual

**Read the Python:** `groupby('h3')` combines events in the same cell. Named aggregations calculate count, maximum, mean and depth. Missing predictors are imputed only in `X`; source measurements stay missing. Negating `score_samples` makes larger scores mean more unusual.

**Look for:** One anomaly score per occupied H3 cell, relative to this feed and these features.

**Pause and try:** Compare a high-score cell with its count and depth. Unusual does not mean a future earthquake is more likely.


In [ ]:
agg = (
    eq.groupby("h3")
      .agg(count=("mag", "size"), max_mag=("mag", "max"), mean_mag=("mag", "mean"), mean_depth=("depth_km", "mean"))
      .reset_index()
)

X = agg[["count", "max_mag", "mean_mag", "mean_depth"]]
X = X.fillna(X.median()).fillna(0)  # model-only imputation; original measurements stay missing
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso.fit(X)
agg["anomaly_score"] = -iso.score_samples(X)  # larger = more unusual
agg.sort_values("anomaly_score", ascending=False).head(10)


### Code guide 5 — Attach scores to map geometry

**Read the Python:** `itertuples` visits the aggregate rows. The H3 boundary is reordered into GeoJSON coordinates, closed, and split at the date line when necessary. Explicit `float` and `int` conversions make values serializable; `None` preserves missing properties as JSON null.

**Look for:** A FeatureCollection with one Polygon or MultiPolygon and a set of model attributes per cell. Pole-enclosing cells fail with an explicit GIS-conversion message.

**Pause and try:** Identify the line that preserves a missing magnitude instead of inventing a measurement.


In [ ]:
from spatial_geometry import split_antimeridian_ring
features = []
for row in agg.itertuples(index=False):
    boundary = h3.cell_to_boundary(row.h3)
    ring = [[lng, lat] for lat, lng in boundary]
    ring.append(ring[0])
    features.append({
        "type": "Feature",
        "properties": {
            "h3": row.h3,
            "count": int(row.count),
            "max_mag": None if pd.isna(row.max_mag) else float(row.max_mag),
            "mean_mag": None if pd.isna(row.mean_mag) else float(row.mean_mag),
            "mean_depth": None if pd.isna(row.mean_depth) else float(row.mean_depth),
            "anomaly_score": float(row.anomaly_score),
        },
        "geometry": split_antimeridian_ring(ring),
    })

h3_fc = {"type": "FeatureCollection", "features": features}


### Code guide 6 — Separate event density from anomaly score

**Read the Python:** The choropleth colors modeled cell scores. The heatmap uses each event with equal weight because `weight_field` is empty. Its radius is a visual smoothing choice, separate from the H3 aggregation scale.

**Look for:** Two layers: where events are concentrated and where cell feature profiles are unusual.

**Pause and try:** Toggle layers and find a place where the two signals disagree.


In [ ]:
m = Map(center=(-160, 15), zoom=2, height="720px")
m.add_choropleth(
    h3_fc,
    column="anomaly_score",
    name="H3 anomaly score",
    class_count=7,
    colormap="turbo",
    scheme="quantile",
    fillOpacity=0.55,
)
m.add_heatmap(
    eq[["lon", "lat"]].to_dict("records"),
    name="Earthquake density (unweighted)",
    radius=22,
    intensity=0.8,
    weight_field="",  # each catalog event contributes equally, including negative/missing magnitudes
)
m


## Model caveat

The Isolation Forest identifies statistical outliers in the chosen features. It does **not** predict earthquakes, estimate seismic hazard, or replace USGS hazard products. This is a visualization/feature-engineering demonstration.

## Data & software citations

- USGS Earthquake Hazards Program GeoJSON feeds: https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php
- Live feed: `https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson`
- H3: https://h3geo.org/ (Apache-2.0).
- scikit-learn Isolation Forest: https://scikit-learn.org/
- GeoLibre: https://geolibre.app/


### Code guide 7 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Connect time, aggregation and anomaly score

The daily bars use UTC dates. The scatter plot pairs the modeled score with the number of cataloged events per H3 cell. Missing magnitudes remain missing; a zero magnitude is a valid measurement, not a missing-value flag.

The density heatmap weights every event equally. Missing aggregate values are median-imputed only in the model matrix; the exported map properties retain nulls.

### Code guide 8 — Check the time window and feature relationships

**Read the Python:** USGS timestamps are milliseconds since the Unix epoch. `utc=True` and `floor('D')` define daily bins consistently. The scatter pairs count with anomaly score and uses color for mean depth; the logarithmic x axis compresses count differences.

**Look for:** Daily event counts, cell scores, feed generation time and a missing-magnitude count.

**Pause and try:** Explain why the first and last day's bars may represent partial days.


In [ ]:
from cognition import style_plots
import matplotlib.pyplot as plt
style_plots()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
times = pd.to_datetime(eq.time, unit="ms", utc=True)
daily = eq.assign(day=times.dt.floor("D")).groupby("day").size()
axes[0].bar(daily.index, daily.values, color="#2563eb")
axes[0].set(xlabel="UTC date (first/last days may be partial)", ylabel="Events", title="Live catalog activity")
axes[0].tick_params(axis="x", rotation=35)
points = axes[1].scatter(agg['count'], agg.anomaly_score, c=agg.mean_depth, cmap="viridis", alpha=.75)
axes[1].set(xscale="log", xlabel="Events in H3 cell (log)", ylabel="Isolation Forest score",
            title="Unusual features are not forecasts")
fig.colorbar(points, ax=axes[1], label="Mean depth (km)")
plt.show()
print("USGS feed generated (UTC):", pd.to_datetime(feed['metadata']['generated'], unit='ms', utc=True))
print("Missing magnitude records:", eq.mag.isna().sum())

**Challenge:** find a high-score, low-count cell. Which other feature could explain its score? Counts also reflect sensor coverage, reporting practices, and the chosen time window.

## Make sense of the result

![A red cell predicts an earthquake? It is unusual in this past-event sample. I will inspect the features, not forecast from the color.](assets/learning/05/cartoon.svg)

**Cartoon transcript:** A red cell predicts an earthquake? → It is unusual in this past-event sample. → I will inspect the features, not forecast from the color.

**Explain it to a colleague:** Can a low-count cell have a high anomaly score?

<details><summary>Check your reasoning</summary>

Yes. Depth or magnitude combinations can differ from the rest even when few events occur.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
